# Section from Robustness_VC_Morgan

In [ ]:
!git clone https://github.com/Plachtaa/seed-vc.git
%cd seed-vc

In [ ]:
# torch --pre --index-url https://download.pytorch.org/whl/nightly/cu126
# torchvision --pre --index-url https://download.pytorch.org/whl/nightly/cu126
# torchaudio --pre --index-url https://download.pytorch.org/whl/nightly/cu126
!pip uninstall -y accelerate
!pip uninstall -y torch
!pip uninstall -y torchvision
!pip uninstall -y torchaudio
!pip uninstall -y scipy
!pip uninstall -y librosa
!pip uninstall -y huggingface-hub
!pip uninstall -y munch
!pip uninstall -y einops
!pip uninstall -y descript-audio-codec
!pip uninstall -y gradio
!pip uninstall -y pydub
!pip uninstall -y resemblyzer
!pip uninstall -y jiwer
!pip uninstall -y transformers
!pip uninstall -y FreeSimpleGUI
!pip uninstall -y soundfile
!pip uninstall -y sounddevice
!pip uninstall -y modelscope
!pip uninstall -y funasr
!pip uninstall -y numpy
!pip uninstall -y hydra-core
!pip uninstall -y pyyaml
!pip uninstall -y python-dotenv
!pip uninstall -y protobuf

In [ ]:
# torch --pre --index-url https://download.pytorch.org/whl/nightly/cu126
# torchvision --pre --index-url https://download.pytorch.org/whl/nightly/cu126
# torchaudio --pre --index-url https://download.pytorch.org/whl/nightly/cu126
!pip install accelerate
!pip install torch==2.4.0
!pip install torchvision==0.19.0ذ
!pip install torchaudio==2.4.0
!pip install scipy==1.13.1
!pip install librosa==0.10.2
!pip install huggingface-hub>=0.28.1
!pip install munch==4.0.0
!pip install einops==0.8.0
!pip install descript-audio-codec==1.0.0
!pip install gradio==5.23.0
!pip install pydub==0.25.1
!pip install resemblyzer
!pip install jiwer==3.0.3
!pip install transformers==4.46.3
!pip install FreeSimpleGUI==5.1.1
!pip install soundfile==0.12.1
!pip install sounddevice==0.5.0
!pip install modelscope==1.18.1
!pip install funasr==1.1.5
!pip install numpy==1.26.4
!pip install hydra-core==1.3.2
!pip install pyyaml
!pip install python-dotenv
!pip install protobuf==5.28.3

In [ ]:
%cd seed-vc

import os
import gc
import random
import numpy as np
import torch
import soundfile as sf

from tqdm import tqdm
from types import SimpleNamespace
from inference_v2 import convert_voice_v2

In [ ]:
import os

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import random
import numpy as np
import torch

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================
# CONFIGURATION
# =====================================

TARGET_SPEAKER = "morgan"

In [ ]:
args = SimpleNamespace(
    diffusion_steps=100,
    length_adjust=1.0,
    intelligibility_cfg_rate=0.7,
    similarity_cfg_rate=0.7,
    top_p=0.9,
    temperature=1.0,
    repetition_penalty=1.0,
    convert_style=False,
    anonymization_only=False,
    compile=False,
    ar_checkpoint_path=None,
    cfm_checkpoint_path=None
)

In [ ]:
# =====================================
# PATHS
# =====================================

ROOT = (
    "/content/drive/MyDrive/"
    "AudioDeepfakesCodes/Data/"
    "Speakers_TrainEvalTest"
)

SOURCE_DIR = os.path.join(
    ROOT,
    "robustness",
    "default",
    "audio"
)

REFERENCE_AUDIO = os.path.join(
    ROOT,
    "robustness",
    "references",
    f"{TARGET_SPEAKER}_neutral_ref.wav"
)

OUTPUT_DIR = os.path.join(
    ROOT,
    "robustness",
    f"vc_{TARGET_SPEAKER}",
    "audio"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [ ]:
# =====================================
# VERIFY REFERENCE EXISTS
# =====================================

if not os.path.exists(
    REFERENCE_AUDIO
):

    raise FileNotFoundError(
        f"Reference audio not found:\n"
        f"{REFERENCE_AUDIO}"
    )

In [ ]:
from IPython.display import Audio

In [ ]:
Audio(REFERENCE_AUDIO)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# =====================================
# GATHER FILES
# =====================================

audio_files = sorted([
    f for f in os.listdir(
        SOURCE_DIR
    )
    if f.endswith(".wav")
])

print(
    f"Target speaker: {TARGET_SPEAKER}"
)

print(
    f"Files found: {len(audio_files)}"
)

print(
    f"Reference: {os.path.basename(REFERENCE_AUDIO)}"
)

In [ ]:
# =====================================
# CONVERT
# =====================================

converted_count = 0
failed_count = 0

failed_files = []

for wav_file in tqdm(
    audio_files,
    desc=f"VC-{TARGET_SPEAKER}"
):

    source_path = os.path.join(
        SOURCE_DIR,
        wav_file
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        wav_file
    )

    # Resume support
    if os.path.exists(
        output_path
    ):
        continue

    try:

        converted_audio = convert_voice_v2(
            source_path,
            REFERENCE_AUDIO,
            args
        )

        save_sr, audio = converted_audio

        sf.write(
            output_path,
            audio,
            save_sr
        )

        converted_count += 1

        if converted_count % 25 == 0:

            print(
                f"\nConverted "
                f"{converted_count} files"
            )

    except Exception as e:

        failed_count += 1

        failed_files.append(
            wav_file
        )

        print(
            f"\nFailed: {wav_file}"
        )

        print(e)

    finally:

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

In [ ]:
# =====================================
# SAVE FAILURE LOG
# =====================================

if len(
    failed_files
) > 0:

    failed_log = os.path.join(
        OUTPUT_DIR,
        "_failed_files.txt"
    )

    with open(
        failed_log,
        "w"
    ) as f:

        for item in failed_files:

            f.write(
                item + "\n"
            )

In [ ]:
# =====================================
# SUMMARY
# =====================================

print("\n========================")

print(
    f"Target speaker: "
    f"{TARGET_SPEAKER}"
)

print(
    f"Converted: "
    f"{converted_count}"
)

print(
    f"Failed: "
    f"{failed_count}"
)

print(
    "\nOutput folder:"
)

print(
    OUTPUT_DIR
)

In [ ]:
import os

ROOT = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness"

default_dir = os.path.join(
    ROOT,
    "default",
    "audio"
)

vc_dir = os.path.join(
    ROOT,
    "vc_david",
    "audio"
)

default_files = {
    f for f in os.listdir(default_dir)
    if f.endswith(".wav")
}

vc_files = {
    f for f in os.listdir(vc_dir)
    if f.endswith(".wav")
}

missing_in_vc = default_files - vc_files
extra_in_vc = vc_files - default_files

print(f"Default files: {len(default_files)}")
print(f"VC files     : {len(vc_files)}")

print(f"\nMissing in vc_david: {len(missing_in_vc)}")
for f in sorted(missing_in_vc):
    print(f)

print(f"\nExtra in vc_david: {len(extra_in_vc)}")
for f in sorted(extra_in_vc):
    print(f)

if len(missing_in_vc) == 0 and len(extra_in_vc) == 0:
    print("\n✅ All filenames match perfectly.")

# Section from Robustness_VC_Jennifer

In [ ]:
# =====================================
# CONFIGURATION
# =====================================

TARGET_SPEAKER = "jennifer"

In [ ]:
# =====================================
# SEED-VC SETTINGS
# =====================================

args = SimpleNamespace(
    diffusion_steps=100,
    length_adjust=1.0,
    intelligibility_cfg_rate=0.8,
    similarity_cfg_rate=0.7,
    top_p=0.9,
    temperature=1.0,
    repetition_penalty=1.0,
    convert_style=False,
    anonymization_only=False,
    compile=False,
    ar_checkpoint_path=None,
    cfm_checkpoint_path=None
)

In [ ]:
# =====================================
# PATHS
# =====================================

ROOT = (
    "/content/drive/MyDrive/"
    "AudioDeepfakesCodes/Data/"
    "Speakers_TrainEvalTest"
)

SOURCE_DIR = os.path.join(
    ROOT,
    "robustness",
    "default",
    "audio"
)

REFERENCE_AUDIO = os.path.join(
    ROOT,
    "robustness",
    "references",
    f"{TARGET_SPEAKER}_neutral_ref2.wav"
)

OUTPUT_DIR = os.path.join(
    ROOT,
    "robustness",
    f"vc_{TARGET_SPEAKER}",
    "audio"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [ ]:
import os

ROOT = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness"

default_dir = os.path.join(
    ROOT,
    "default",
    "audio"
)

vc_dir = os.path.join(
    ROOT,
    "vc_david",
    "audio"
)

default_files = {
    f for f in os.listdir(default_dir)
    if f.endswith(".wav")
}

vc_files = {
    f for f in os.listdir(vc_dir)
    if f.endswith(".wav")
}

missing_in_vc = default_files - vc_files
extra_in_vc = vc_files - default_files

print(f"Default files: {len(default_files)}")
print(f"VC files     : {len(vc_files)}")

print(f"\nMissing in vc_jennifer: {len(missing_in_vc)}")
for f in sorted(missing_in_vc):
    print(f)

print(f"\nExtra in vc_jennifer: {len(extra_in_vc)}")
for f in sorted(extra_in_vc):
    print(f)

if len(missing_in_vc) == 0 and len(extra_in_vc) == 0:
    print("\n✅ All filenames match perfectly.")

# Section from Robustness_VC_David

In [ ]:
# =====================================
# CONFIGURATION
# =====================================

TARGET_SPEAKER = "david"

In [ ]:
# =====================================
# SEED-VC SETTINGS
# =====================================

args = SimpleNamespace(
    diffusion_steps=100,
    # diffusion_steps=150,
    length_adjust=1.0,
    # intelligibility_cfg_rate=0.7,
    intelligibility_cfg_rate=1.2,
    # similarity_cfg_rate=0.7,
    similarity_cfg_rate=1.1,
    top_p=0.9,
    temperature=1.2,
    repetition_penalty=1.2,
    # convert_style=False,
    convert_style=False,
    anonymization_only=False,
    compile=False,
    ar_checkpoint_path=None,
    cfm_checkpoint_path=None
)
